# Cyber expert training

In [1]:
!nvidia-smi

Sat Jan 10 01:42:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 1: Setup Environment with uv
Instead of standard pip, we use uv to ensure we follow the uv.lock file you generated on your Mac.

In [4]:
# 1. Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] += ":/root/.cargo/bin"

# 2. Install dependencies into the Colab system environment
# We use --system because Colab handles its own Python path
print("Installing expert-factory dependencies...")
!uv pip install --system .

downloading uv 0.9.24 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Installing expert-factory dependencies...
Using Python 3.12.12 environment at: /usr
error: /content does not appear to be a Python project, as neither `pyproject.toml` nor `setup.py` are present in the directory


### 2: Initialize Model (Gemma 2 9B)

We load the model in 4-bit to fit into the T4's 16GB VRAM.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-9b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4: Load and Format Expert Data

This cell loads the Cybersecurity datasets and applies the Gemma chat template.

In [ ]:
from datasets import load_dataset

# Loading the Cybersecurity dataset
dataset = load_dataset("ethanolivertroy/nist-cybersecurity-training", split = "train")


# Define the Expert Persona
SYSTEM_PROMPT = """
You are a Senior Cybersecurity Analyst and Threat Hunter. Provide technical, accurate,
and actionable security advice.
"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["response"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        # Apply the Gemma 2 Chat Template
        text = f"""
            <start_of_turn>user\n{SYSTEM_PROMPT}\n\n{instruction}<end_of_turn>\n
            <start_of_turn>model\n{output}<end_of_turn>
            """
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

## 5: Training Configuration (T4 Optimized)

These settings prevent "Out of Memory" (OOM) errors on the T4 GPU.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100, # Set to ~300-500 for a more complete expert
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "models/adapters/cybersecurity", # Save to your project folder
    ),
)

# Start training
trainer_stats = trainer.train()

## 6: Expert Inference Test

Test the model inside the notebook to see if it adopts the expert persona.

In [ ]:
FastLanguageModel.for_inference(model) # 2x faster inference

prompt = "Explain the security implications of finding an exposed .env file on a public web server."

inputs = tokenizer(
[
    f"<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

## 7: Export to GGUF (For LM Studio/Ollama)

This merges the LoRA adapter into the base model and quantizes it into a single file.

In [ ]:
# Save the final GGUF model directly to your Google Drive
model.save_pretrained_gguf(
    "models/final_gguf/cyber_expert_gemma",
    tokenizer,
    quantization_method = "q4_k_m"
)